# 2018–2020 韩国关税署 HWP 原稿表格候选数值检查

同一 OLE/压缩正文方法先复核2021 PDF与2022 HWP，再提取2018–2020每年的半导体表格候选；有歧义不自动定值。

In [1]:
import hashlib,json,re,zlib
from pathlib import Path
import olefile
root=Path.cwd();card=root/'research/kcs_early20_2018_2020_oct_original_attachments_20261023';plan=json.loads((card/'hwp_numeric_review_plan.json').read_text());source=json.loads((card/'result.json').read_text());sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and sha(card/'result.json')==plan['source_result_sha256'] and source['NAS_source_id']==plan['NAS_source_id'] and source['NAS_readback_verified'] and olefile.__version__=='0.47'
raw=root/'data/snapshots/kcs_early20_2018_2020_oct_original_attachments_20261023'
for year,digest in plan['source_HWP_hashes'].items():assert sha(raw/(year+'_attachment.hwp'))==digest
control_paths={2021:root/'data/snapshots/kcs_early20_2021_oct_original_source_20261023/2021_early20.hwp',2022:root/'data/snapshots/kcs_early20_2022_oct_original_hwp_20261023/2022_early20.hwp'}
def read(path):
    with olefile.OleFileIO(path) as f:
        assert f.exists('PrvText') and f.exists('BodyText/Section0')
        preview=f.openstream('PrvText').read().decode('utf-16le','replace');body=zlib.decompress(f.openstream('BodyText/Section0').read(),-15);meta=f.get_metadata()
        return preview,body,{'created':str(meta.create_time) if meta.create_time else None,'last_saved':str(meta.last_saved_time) if meta.last_saved_time else None,'OLE_streams':len(f.listdir())}
def spans(data,expression):
    found=[]
    for shift in (0,1):
        text=data[shift:].decode('utf-16le','ignore')
        for m in re.finditer(expression,text):found.append({'byte_offset':shift+2*m.start(),'text':m.group(),'shift':shift})
    return sorted(found,key=lambda x:(x['byte_offset'],x['text']))
def review(path):
    preview,body,meta=read(path)
    labels=spans(body,'반도체');amounts=spans(body,r'(?<!\d)\d{1,3}(?:,\d{3})+(?!\d)');yoys=spans(body,r'(?<!\d)\d{1,3}\.\d(?!\d)')
    neighborhoods=[]
    for lab in labels:
        near=[a for a in amounts if 0<a['byte_offset']-lab['byte_offset']<=650]
        pairs=[]
        for a in near:
            for y in yoys:
                if 0<y['byte_offset']-a['byte_offset']<=350:pairs.append({'amount':a['text'],'amount_byte_offset':a['byte_offset'],'yoy_abs':y['text'],'yoy_byte_offset':y['byte_offset']})
        if pairs:neighborhoods.append({'label_byte_offset':lab['byte_offset'],'pairs':pairs[:30]})
    return {'preview_chars':len(preview),'preview_excerpt':preview[:1700],'body_bytes':len(body),'OLE_metadata':meta,'semiconductor_label_offsets':sorted({z['byte_offset'] for z in labels}),'nearby_numeric_pairs':neighborhoods,'unit_token_offsets':sorted({z['byte_offset'] for z in spans(body,'백만 달러')})[:12]}
controls={str(y):review(p) for y,p in control_paths.items()}
assert any(z['amount']=='6,394' and z['yoy_abs']=='23.9' for n in controls['2021']['nearby_numeric_pairs'] for z in n['pairs'])
assert any(z['amount']=='5,573' and z['yoy_abs']=='12.8' for n in controls['2022']['nearby_numeric_pairs'] for z in n['pairs'])
unknown={year:review(raw/(year+'_attachment.hwp')) for year in plan['source_HWP_hashes']}
result={'decision':'HWP_binary_numeric_candidates_extracted_review_required','plan_sha256':sha(card/'hwp_numeric_review_plan.json'),'source_result_sha256':plan['source_result_sha256'],'control_2021_2022_passed':True,'years':unknown,'controls':controls,'source_HWP_reads':3,'control_HWP_reads':2,'external_GETs':0,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0,'historical_first_public_bytes_qualified':False,'full_monthly_training_panel_qualified':False}
(card/'hwp_numeric_probe.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print(json.dumps({'decision':result['decision'],'control_passed':True,'unknown_pairs':{year:data['nearby_numeric_pairs'] for year,data in unknown.items()}},ensure_ascii=False))


{"decision": "HWP_binary_numeric_candidates_extracted_review_required", "control_passed": true, "unknown_pairs": {"2018": [], "2019": [], "2020": []}}


初次原始字节邻近搜索通过2021/2022控制稿，但2018–2020表格候选无唯一金额；值保持未知。见 hwp_numeric_probe.json。
